# D — mT5-small Kaggle GPU smoke test (Week 4)

Run this on a Kaggle or Colab **GPU** notebook. It loads `google/mt5-small` locally, runs one forward pass, and records memory. No hosted LLM API calls.

Enable GPU: Kaggle → Settings → Accelerator → GPU T4.

In [ ]:
!pip -q install transformers==4.44.2 sentencepiece accelerate
import json, time, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

report = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
print(report)

In [ ]:
MODEL = "google/mt5-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
model.eval()
load_s = round(time.time() - t0, 2)
params = sum(p.numel() for p in model.parameters())
print("params", params, "load_s", load_s)

In [ ]:
prompt = "STANDARD context: lecture ending soon sia | yeah I'm starving message: eh later go makan? can then library"
batch = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64).to(device)
with torch.no_grad():
    t1 = time.time()
    _ = model.generate(**batch, max_new_tokens=24)
    gen_s = round(time.time() - t1, 2)

peak_gb = None
if torch.cuda.is_available():
    peak_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3)

report.update({
    "model": MODEL,
    "parameters": int(params),
    "load_seconds": load_s,
    "generate_seconds": gen_s,
    "max_length": 64,
    "max_new_tokens": 24,
    "peak_cuda_gb": peak_gb,
    "api_calls": 0,
})
print(json.dumps(report, indent=2))
with open("mt5_smoke_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("wrote mt5_smoke_report.json — copy this into the team repo experiments/logs/")